# Simple CNN for MNIST (PyTorch)

## basic CNN with around ~97% test accuracy after just 3 epochs.

how it works

we take input 28x28 grayscale images

apply 1 single convolution (8 filters)

apply ReLU → MaxPool → so image becomes 14x14

flatten and pass to 1 Linear layer → 10 classes

we train using:

CrossEntropyLoss

SGD (lr=0.01)

3 epochs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(8*14*14, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# load MNIST
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=".", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

# model, loss, optimiser
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# training loop
for epoch in range(3):
    total = 0
    correct = 0
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"epoch {epoch+1}: accuracy = {correct/total:.4f}")

# test accuracy
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("test accuracy:", correct/total)
